# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [1]:
# TODO


## 2. Ressources (psutil)

In [8]:
"""Volet ressources — mesures psutil sur le modèle legacy (RSS, temps, taille, débit)."""
import platform, statistics as stats, subprocess, sys, time
from pathlib import Path

import joblib
import pandas as pd
import psutil
import sklearn
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL = ROOT / "legacy" / "dms_predictor_v1.joblib"
DATA = ROOT / "data" / "dms_dataset.csv"

df = pd.read_csv(DATA)
X = df[["age", "nb_comorbidites", "imc"]].copy()
X["sexe_bin"] = (df["sexe"] == "M").astype(int)
y = df["sejour_prolonge"]

BANC = (f"{platform.system()} {platform.machine()} | Python {platform.python_version()} | "
        f"sklearn {sklearn.__version__} | {psutil.cpu_count(logical=False)} cœurs physiques / "
        f"{psutil.cpu_count()} logiques | {psutil.virtual_memory().total / 1e9:.1f} Go RAM")
print(BANC)

# --- RSS isolé : le kernel a déjà tout importé, on mesure donc dans un process neuf ---
# Le prélude reproduit les imports de predict.py ; sklearn n'y figure pas : c'est joblib.load qui le tire.
PROBE = """
import os, sys, time, psutil, joblib, pandas
{prelude}
p = psutil.Process(os.getpid())
base = p.memory_info().rss
t0 = time.perf_counter(); m = joblib.load(sys.argv[1]); t = time.perf_counter() - t0
print(base, p.memory_info().rss, t)
"""
SKL = "import sklearn.ensemble, sklearn.linear_model"


def probe(path, prelude="", n=5):
    """RSS avant/après joblib.load dans un process neuf ; médiane sur n runs (le 1er paie le cache disque)."""
    rows = []
    for _ in range(n):
        out = subprocess.run([sys.executable, "-c", PROBE.format(prelude=prelude), str(path)],
                             capture_output=True, text=True, cwd=ROOT, check=True)
        b, a, t = out.stdout.split()
        rows.append((int(b) / 1e6, int(a) / 1e6, float(t) * 1000))
    return tuple(stats.median(c) for c in zip(*rows))


# Sans prélude : `joblib.load` tire l'import de sklearn -> mesure le coût réel de predict.py.
# Avec prélude : sklearn est déjà en mémoire -> isole le modèle seul (mémoire) et la désérialisation (temps).
rss_imports, rss_prod, t_load_ms = probe(MODEL)
rss_skl, rss_skl_model, t_deser_ms = probe(MODEL, prelude=SKL)
rss_modele = rss_skl_model - rss_skl

# --- Latences ---
model = joblib.load(MODEL)
x1 = X.iloc[[0]]
model.predict_proba(x1)                                     # warm-up (allocation numpy)
runs = []
for _ in range(200):
    t0 = time.perf_counter(); model.predict_proba(x1); runs.append((time.perf_counter() - t0) * 1000)
lat_1 = stats.median(runs)

batch = []
for _ in range(5):
    t0 = time.perf_counter(); model.predict_proba(X); batch.append((time.perf_counter() - t0) * 1000)
lat_batch = stats.median(batch)

# --- Latence réelle de production : 1 process python complet par patient ---
cold = []
for _ in range(5):
    t0 = time.perf_counter()
    subprocess.run([sys.executable, "legacy/predict.py", "70", "3", "28.5", "1"],
                   cwd=ROOT, capture_output=True)
    cold.append((time.perf_counter() - t0) * 1000)
lat_cold = stats.median(cold)

# --- Temps de ré-entraînement (mêmes hyperparamètres que legacy/train.py) ---
fits = []
for _ in range(3):
    t0 = time.perf_counter()
    RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0).fit(X, y)
    fits.append(time.perf_counter() - t0)
t_fit = stats.median(fits)

size_mo = MODEL.stat().st_size / 1e6
n_leaves = sum(e.get_n_leaves() for e in model.estimators_)

print(f"""
Taille artefact              : {size_mo:.2f} Mo ({MODEL.stat().st_size} o) — {n_leaves} feuilles

MÉMOIRE (RSS, process neuf, médiane sur 5 runs)
  après imports predict.py   : {rss_imports:.1f} Mo   (python + pandas + joblib)
  + import sklearn           : {rss_skl:.1f} Mo   -> sklearn seul : +{rss_skl - rss_imports:.1f} Mo
  + le modèle                : {rss_skl_model:.1f} Mo   -> MODÈLE SEUL : +{rss_modele:.1f} Mo ({rss_modele / size_mo:.1f}x son fichier)
  RSS total d'un appel prod  : {rss_prod:.1f} Mo   (dont {(1 - rss_modele / rss_prod) * 100:.0f} % = runtime, pas le modèle)

TEMPS
  joblib.load (process neuf) : {t_load_ms:.0f} ms  dont désérialisation pure : {t_deser_ms:.0f} ms
                               -> import sklearn tiré par le load : {t_load_ms - t_deser_ms:.0f} ms
  Fit (10 000 lignes)        : {t_fit * 1000:.0f} ms
  Inférence 1 ligne (chaud)  : {lat_1:.2f} ms     -> {1000 / lat_1:.0f} pred/s
  Inférence batch 10 000     : {lat_batch:.1f} ms -> {10_000 / (lat_batch / 1000):,.0f} pred/s ({lat_batch * 1000 / 10_000:.1f} us/pred)
  Appel prod (predict.py)    : {lat_cold:.0f} ms  -> {1000 / lat_cold:.2f} pred/s
      calcul utile           : {lat_1 / lat_cold * 100:.2f} %
      chargement du modèle   : {t_load_ms / lat_cold * 100:.1f} %  (dont {t_deser_ms / lat_cold * 100:.1f} % de désérialisation)
      démarrage + imports    : {(lat_cold - lat_1 - t_load_ms) / lat_cold * 100:.1f} %
  10 000 prédictions         : {lat_batch / 1000:.2f} s en batch  vs  {lat_cold * 10_000 / 3.6e6:.2f} h en mode actuel (x{lat_cold * 10_000 / lat_batch:,.0f})
""")

LEGACY = {"nom": "legacy_rf", "taille_mo": size_mo, "rss_modele_mo": rss_modele,
          "t_load_ms": t_load_ms, "t_deser_ms": t_deser_ms, "t_fit_s": t_fit,
          "lat_1ligne_ms": lat_1, "lat_batch10k_ms": lat_batch, "lat_prod_ms": lat_cold}


Windows AMD64 | Python 3.11.15 | sklearn 1.5.1 | 10 cœurs physiques / 12 logiques | 34.0 Go RAM

Taille artefact              : 4.96 Mo (4956361 o) — 30846 feuilles

MÉMOIRE (RSS, process neuf, médiane sur 5 runs)
  après imports predict.py   : 72.7 Mo   (python + pandas + joblib)
  + import sklearn           : 141.5 Mo   -> sklearn seul : +68.8 Mo
  + le modèle                : 148.5 Mo   -> MODÈLE SEUL : +7.1 Mo (1.4x son fichier)
  RSS total d'un appel prod  : 148.6 Mo   (dont 95 % = runtime, pas le modèle)

TEMPS
  joblib.load (process neuf) : 918 ms  dont désérialisation pure : 15 ms
                               -> import sklearn tiré par le load : 903 ms
  Fit (10 000 lignes)        : 337 ms
  Inférence 1 ligne (chaud)  : 1.51 ms     -> 664 pred/s
  Inférence batch 10 000     : 52.0 ms -> 192,482 pred/s (5.2 us/pred)
  Appel prod (predict.py)    : 1890 ms  -> 0.53 pred/s
      calcul utile           : 0.08 %
      chargement du modèle   : 48.6 %  (dont 0.8 % de désérialisation)

## 3. Comparaison à 2 alternatives

`logreg` (régression logistique) et `histgb` (boosting), plus la variante `logreg_sans_sexe` qui sert
de contrefactuel au volet éthique. Protocole identique pour tous : mêmes données, mêmes folds, même sonde RSS.


In [9]:
"""Comparaison sobriété : legacy vs alternatives légères, à données et protocole identiques."""
import tempfile

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
TMP = Path(tempfile.mkdtemp())
Y_REF = (df["dms_jours"] >= 5.6).astype(int).values   # règle d'étiquetage reconstruite (cf. 01_ethique § 2.3)
EST_F = (df["sexe"] == "F").values


def equite(pred, proba):
    """DI F/M sur les prédictions + FNR par sexe mesuré contre la référence y_ref."""
    di = pred[EST_F].mean() / pred[~EST_F].mean()
    fnr_f = 1 - pred[EST_F & (Y_REF == 1)].mean()
    fnr_m = 1 - pred[~EST_F & (Y_REF == 1)].mean()
    return di, fnr_f, fnr_m, proba[EST_F].mean(), proba[~EST_F].mean()


def mesurer(nom, est, Xi):
    """Fit + taille + RSS modèle seul + latences + qualité CV + équité — protocole identique pour tous."""
    fits = []
    for _ in range(3):
        t0 = time.perf_counter(); est.fit(Xi, y); fits.append(time.perf_counter() - t0)

    path = TMP / f"{nom}.joblib"
    joblib.dump(est, path)
    rss_b, rss_a, t_deser = probe(path, prelude=SKL, n=3)   # sklearn préchargé -> RSS du modèle seul

    xi1 = Xi.iloc[[0]]
    est.predict_proba(xi1)                                   # warm-up
    r = [(lambda t0: (est.predict_proba(xi1), (time.perf_counter() - t0) * 1000)[1])(time.perf_counter())
         for _ in range(200)]
    b = [(lambda t0: (est.predict_proba(Xi), (time.perf_counter() - t0) * 1000)[1])(time.perf_counter())
         for _ in range(5)]

    sc = cross_validate(est, Xi, y, cv=CV, scoring=["accuracy", "f1", "roc_auc"])
    proba = est.predict_proba(Xi)[:, 1]
    di, fnr_f, fnr_m, p_f, p_m = equite((proba >= 0.5).astype(int), proba)
    return {"nom": nom, "t_fit_s": stats.median(fits), "taille_ko": path.stat().st_size / 1e3,
            "rss_modele_mo": rss_a - rss_b, "t_deser_ms": t_deser,
            "lat1_ms": stats.median(r), "latb_ms": stats.median(b),
            "acc": sc["test_accuracy"].mean(), "f1": sc["test_f1"].mean(), "auc": sc["test_roc_auc"].mean(),
            "di": di, "fnr_f": fnr_f, "fnr_m": fnr_m, "p_f": p_f, "p_m": p_m}


X_ns = X.drop(columns=["sexe_bin"])                          # variante : mêmes features, sans le sexe
logreg = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

R = pd.DataFrame([
    mesurer("legacy_rf", RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0), X),
    mesurer("logreg", logreg(), X),
    mesurer("histgb", HistGradientBoostingClassifier(max_iter=100, random_state=0), X),
    mesurer("logreg_sans_sexe", logreg(), X_ns),
]).set_index("nom")

ref = R.loc["legacy_rf"]
R["taille_x"] = (ref.taille_ko / R.taille_ko).round(0)
R["fit_x"] = (ref.t_fit_s / R.t_fit_s).round(1)
R["rss_x"] = (ref.rss_modele_mo / R.rss_modele_mo).round(1)
R["latb_x"] = (ref.latb_ms / R.latb_ms).round(1)

pd.set_option("display.width", 220, "display.max_columns", 30)
print(f"Banc : {BANC}\n")
print("=== Ressources (legacy_rf = référence) ===")
print(R[["taille_ko", "rss_modele_mo", "t_deser_ms", "t_fit_s", "lat1_ms", "latb_ms",
         "taille_x", "fit_x", "rss_x", "latb_x"]].round(3).to_string())
print("\n=== Qualité (CV 5 folds stratifiés, shuffle, random_state=42) ===")
print(R[["acc", "f1", "auc"]].round(4).to_string())
print("\n=== Équité (in-sample, même protocole que 01_ethique — DI cible ≥ 0,80) ===")
print(R[["di", "fnr_f", "fnr_m", "p_f", "p_m"]].round(4).to_string())


Banc : Windows AMD64 | Python 3.11.15 | sklearn 1.5.1 | 10 cœurs physiques / 12 logiques | 34.0 Go RAM

=== Ressources (legacy_rf = référence) ===
                  taille_ko  rss_modele_mo  t_deser_ms  t_fit_s  lat1_ms  latb_ms  taille_x  fit_x  rss_x  latb_x
nom                                                                                                              
legacy_rf          4959.529          7.053      24.666    0.338    1.524   44.476       1.0    1.0    1.0     1.0
logreg                1.822          0.025       1.533    0.010    1.260    3.670    2722.0   33.0  287.0    12.1
histgb              366.080          0.737       9.013    0.508    2.630   21.786      14.0    0.7    9.6     2.0
logreg_sans_sexe      1.758          0.029       3.285    0.007    0.389    2.859    2821.0   50.6  246.0    15.6

=== Qualité (CV 5 folds stratifiés, shuffle, random_state=42) ===
                     acc      f1     auc
nom                                     
legacy_rf         0.

## 4. Coût du modèle — ordre de grandeur

Toutes les hypothèses sont **explicites et paramétrables** ci-dessous. L'objectif n'est pas un chiffrage
comptable mais un **ordre de grandeur** permettant de comparer les postes entre eux.


In [10]:
"""Coût du modèle — ordre de grandeur, hypothèses explicites (à valider avec Hélène : Q_vol, Q_geste)."""
# --- Hypothèses (à confirmer par MediVox) ---
H = {
    "sejours_an": 10_000,      # = taille du dataset ; volume réel INCONNU -> Q16
    "geste_min": 2,            # minutes humaines par appel SSH (connexion, saisie, lecture, report) -> Q17
    "cout_horaire_eur": 45,    # coût chargé d'un profil soignant/administratif
    "puissance_w": 25,         # puissance moyenne d'un cœur x86 serveur sous charge
    "prix_kwh_eur": 0.174,     # tarif pro France 2025
    "kgco2_kwh": 0.056,        # mix électrique français (ordre de grandeur ADEME/RTE)
    "serveur_eur_an": 600,     # VM ou amortissement serveur dédié, 24/7
}

n = H["sejours_an"]
cpu_h_actuel = lat_cold * n / 3.6e6                       # 1 process python par patient
cpu_h_batch = (lat_batch / 1000 + lat_cold / 1000) / 3600  # 1 seul démarrage + 1 batch
kwh = cpu_h_actuel * H["puissance_w"] / 1000
h_humaines = n * H["geste_min"] / 60

postes = pd.DataFrame([
    ("Calcul — mode actuel (1 process/patient)", cpu_h_actuel * H["puissance_w"] / 1000 * H["prix_kwh_eur"]),
    ("Calcul — même volume en batch", cpu_h_batch * H["puissance_w"] / 1000 * H["prix_kwh_eur"]),
    ("Ré-entraînement (1x/an)", t_fit * H["puissance_w"] / 3.6e6 * H["prix_kwh_eur"]),
    ("Serveur dédié 24/7 (coût fixe)", H["serveur_eur_an"]),
    ("Geste humain SSH", h_humaines * H["cout_horaire_eur"]),
], columns=["poste", "eur_an"]).set_index("poste")
postes["part_%"] = (postes.eur_an / postes.eur_an.sum() * 100).round(2)

print(f"""HYPOTHÈSES : {n:,} séjours/an · {H['geste_min']} min de geste humain/appel · {H['cout_horaire_eur']} €/h
             {H['puissance_w']} W · {H['prix_kwh_eur']} €/kWh · {H['kgco2_kwh']} kgCO2e/kWh · serveur {H['serveur_eur_an']} €/an

CALCUL
  CPU mode actuel   : {cpu_h_actuel:.2f} h/an  -> {kwh:.3f} kWh  -> {kwh * H['kgco2_kwh'] * 1000:.0f} gCO2e/an
  CPU en batch      : {cpu_h_batch * 3600:.1f} s/an ({cpu_h_actuel * 3600 / (cpu_h_batch * 3600):,.0f}x moins)
  Taux d'occupation du serveur : {cpu_h_actuel / 8760 * 100:.3f} % de l'année
  Ré-entraînement   : {t_fit:.2f} s de CPU, 1 fois -> coût électrique {t_fit * H['puissance_w'] / 3.6e6 * H['prix_kwh_eur']:.2e} €

HUMAIN
  Geste SSH         : {h_humaines:,.0f} h/an, soit {h_humaines / 1607:.1f} ETP

POSTES (€/an, ordre de grandeur)""")
print(postes.round(2).to_string())
print(f"""
LECTURE : le calcul coûte {postes.loc['Calcul — mode actuel (1 process/patient)', 'eur_an']:.2f} €/an.
Le geste humain coûte {postes.loc['Geste humain SSH', 'eur_an'] / max(postes.loc['Calcul — mode actuel (1 process/patient)', 'eur_an'], 1e-9):,.0f}x plus cher.
Empreinte carbone annuelle du calcul : {kwh * H['kgco2_kwh'] * 1000:.0f} gCO2e — soit ~{kwh * H['kgco2_kwh'] / 0.103 * 1000:.0f} m parcourus en voiture thermique.
Avec logreg, l'artefact passerait de {size_mo * 1000:.0f} Ko à {R.loc['logreg', 'taille_ko']:.1f} Ko : gain réel = 0 € — le modèle n'a JAMAIS été le coût.
""")


HYPOTHÈSES : 10,000 séjours/an · 2 min de geste humain/appel · 45 €/h
             25 W · 0.174 €/kWh · 0.056 kgCO2e/kWh · serveur 600 €/an

CALCUL
  CPU mode actuel   : 5.25 h/an  -> 0.131 kWh  -> 7 gCO2e/an
  CPU en batch      : 1.9 s/an (9,733x moins)
  Taux d'occupation du serveur : 0.060 % de l'année
  Ré-entraînement   : 0.34 s de CPU, 1 fois -> coût électrique 4.07e-07 €

HUMAIN
  Geste SSH         : 333 h/an, soit 0.2 ETP

POSTES (€/an, ordre de grandeur)
                                            eur_an  part_%
poste                                                     
Calcul — mode actuel (1 process/patient)      0.02    0.00
Calcul — même volume en batch                 0.00    0.00
Ré-entraînement (1x/an)                       0.00    0.00
Serveur dédié 24/7 (coût fixe)              600.00    3.85
Geste humain SSH                          15000.00   96.15

LECTURE : le calcul coûte 0.02 €/an.
Le geste humain coûte 656,644x plus cher.
Empreinte carbone annuelle du calcul : 